# Create Delta Table

1.In previous Databricks versions, DBFS was available and commonly used. In the current environment, DBFS Root is disabled, and Unity Catalog Volumes are the recommended storage option. Volumes work directly with SQL using the /Volumes/<catalog>/<schema>/<volume>/ path. If you want to access the same storage from Python APIs that require a cloud storage URI, use the underlying container-level path (for example, an abfss://, s3://, or gs:// URL) instead of the /Volumes/... path.

2.If you cannot create a Delta table using an external location, you cannot access it by path. Use the table name instead.


In [0]:
%sql
CREATE OR REPLACE TABLE tri.tri.employee_demo (
    emp_id INT,
    emp_name STRING,
    gender STRING,
    salary INT,
    Dept STRING
)
USING DELTA
LOCATION 'dbfs:/Volumes/tri/tri/tri_files/tables/employee_demo';

### #above one not works

In [0]:
%sql
CREATE OR REPLACE TABLE tri.tri.employee_demo (
    emp_id INT,
    emp_name STRING,
    gender STRING,
    salary INT,
    dept STRING
)
USING DELTA;

In [0]:
%sql
insert into tri.tri.employee_demo values(100,"Stephen","M",2000,"IT");
insert into tri.tri.employee_demo values(200,"Philipp","M",8000,"HR");
insert into tri.tri.employee_demo values(300,"Lara","F",6000,"SALES");
insert into tri.tri.employee_demo values(400,"Mike","M",4000,"IT");
insert into tri.tri.employee_demo values(500,"Sarah","F",9000,"HR");
insert into tri.tri.employee_demo values(600,"Serena","F",5000,"SALES");
insert into tri.tri.employee_demo values(700,"Mark","M",7000,"SALES");

In [0]:
%sql
select * from tri.tri.employee_demo;

# Method 1: SQL Standard using Table Name

In [0]:
%sql
UPDATE tri.tri.employee_demo
SET salary = 10000
WHERE emp_name = 'Mark';

In [0]:
%sql
SELECT * FROM tri.tri.employee_demo;

# Method 2: SQL Standard using Table Path

In [0]:
%sql
UPDATE delta.`/FileStore/tables/delta/path_employee_demo`
SET salary = 12000
WHERE emp_name = 'Mark';

In [0]:
%sql
SELECT * FROM employee_demo;

# Method 3: DeltaTable Update Using SQL Predicate (Table Instance)

In [0]:
from delta.tables import *
from pyspark.sql.functions import *
deltaTable = DeltaTable.forPath(spark, '/Volumes/tri/tri/tri_files/tables/employee_demo')


# Declare the predicate by using a SQL-formatted string.
deltaTable.update(
    condition = "emp_name = 'Mark'",
    set = { "salary": "15000" }
)

### #above not works

In [0]:
from delta.tables import *
from pyspark.sql.functions import *

deltaTable = DeltaTable.forName(spark, 'tri.tri.employee_demo')

# Declare the predicate by using a SQL-formatted string.
deltaTable.update(
    condition = "emp_name = 'Mark'",
    set = { "salary": "15000" }
)

In [0]:
%sql
select * from tri.tri.employee_demo

Description
- •	Uses a SQL predicate specified as a string.
- •	Similar to a SQL WHERE clause.
- •	Commonly used when conditions are easy to express in SQL syntax.


# Method 4 - DeltaTable Update Using Spark Column Expressions (Table Instance)

In [0]:
# Declare the predicate by using Spark SQL functions.
deltaTable.update(
    condition = col('emp_name') == 'Mark',
    set = { 'Dept': lit('IT') }
)

In [0]:
%sql
select * from employee_demo

Description
- •	Uses Spark Column Expressions (col(), lit()).
- •	Programmatic approach using the PySpark API.
- •	Useful for dynamic conditions and complex logic.
